# 16. 리뷰 10개 미만 게임의 실패 유형 스토리라인 분석

**분석 목적:** `steam_indie_games_silence.csv`에 포함된 리뷰 10개 미만 게임을 대상으로, PPT에서 사용할 수 있는 “왜 실패했는가” 분석 흐름을 구성한다.

**핵심 관점:** 리뷰 10개 미만 게임은 단순히 유저가 싫어한 게임이라기보다, 시장에서 충분한 초기 반응을 확보하지 못한 **무반응/침묵 게임**으로 해석한다.

## 분석 흐름

1. 실패 기준 정의: 리뷰 10개 미만을 무반응 게임으로 정의
2. 유저 반응 정도 구분: 무반응, 약한 반응, 기준 근접 그룹 분리
3. 출시 시점: 연도·월별 무반응 게임 분포 확인
4. 장르: 어떤 장르 조합에서 침묵 게임이 많이 관측되는지 확인
5. 가격: 가격대별로 리뷰 확보 정도가 어떻게 다른지 확인
6. 태그: 실패 게임에서 반복되는 태그와 차별화 부족 신호 확인
7. 태그 조합: 리뷰 10개 이상 반응을 이끌어낸 2개/3개 태그 조합과 통계적 신뢰도 확인
8. 상점 신뢰 신호: 도전과제, 플랫폼, 컨트롤러, 카테고리, 설명 길이 비교
9. 실패 유형화: 노출 실패형, 차별화 부족형, 가격/기대 불일치형, 신뢰 신호 부족형으로 정리

In [141]:
import ast
import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# VS Code/JupyterLab에서는 plotly_mimetype, classic notebook에서는 notebook_connected로 inline 표시한다.
pio.renderers.default = "plotly_mimetype+notebook_connected"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

COLOR_MAIN = "#3A6EA5"
COLOR_ACCENT = "#D97A3A"
COLOR_MUTED = "#6B7280"
COLOR_RISK = "#B94E48"
COLOR_GOOD = "#4F8A5B"


def find_data_path(filename: str) -> Path:
    candidates = [
        Path("../../../data/preprocessed") / filename,
        Path("data/preprocessed") / filename,
        Path("../../data/preprocessed") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"데이터 파일을 찾을 수 없습니다: {filename}")


def parse_list_literal(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (SyntaxError, ValueError):
            return []
    return []


def parse_tag_names(value):
    if isinstance(value, dict):
        return list(value.keys())
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        for loader in (json.loads, ast.literal_eval):
            try:
                parsed = loader(value)
                if isinstance(parsed, dict):
                    return list(parsed.keys())
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                continue
    return []


def split_categories(value):
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def to_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() == "true"


def make_price_tier(price):
    if pd.isna(price):
        return "가격 정보 없음"
    if price <= 0:
        return "무료"
    if price < 5:
        return "~$5"
    if price < 10:
        return "$5~$10"
    if price < 15:
        return "$10~$15"
    if price < 20:
        return "$15~$20"
    return "$20+"


def make_review_band(total_reviews):
    if total_reviews == 0:
        return "무반응(리뷰 0개)"
    if total_reviews <= 3:
        return "미약 반응(리뷰 1~3개)"
    if total_reviews <= 6:
        return "제한 반응(리뷰 4~6개)"
    return "반응 임계권(리뷰 7~9개)"

SILENCE_PATH = find_data_path("steam_indie_games_silence.csv")
RESPONSE_PATH = find_data_path("steam_indie_games.csv")

df = pd.read_csv(SILENCE_PATH)
df_response = pd.read_csv(RESPONSE_PATH)

population_df = pd.concat(
    [
        df[["appid", "total_reviews"]].assign(response_group="무반응(리뷰 10개 미만)"),
        df_response[["appid", "total_reviews"]].assign(response_group="반응 확보(리뷰 10개 이상)"),
    ],
    ignore_index=True,
).drop_duplicates("appid", keep="first")

# 파생 변수 생성
for col in ["windows", "mac", "linux"]:
    df[col] = df[col].apply(to_bool)

df["genre_list"] = df["genres"].apply(parse_list_literal)
df["tag_list"] = df["tags"].apply(parse_tag_names)
df["category_list"] = df["categories"].apply(split_categories)
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month
df["review_band"] = df["total_reviews"].apply(make_review_band)
df["price_tier"] = df["price"].apply(make_price_tier)
df["positive_rate"] = np.where(df["total_reviews"] > 0, df["positive"] / df["total_reviews"], np.nan)
df["description_len"] = df["short_description"].fillna("").str.len()
df["genre_count"] = df["genre_list"].str.len()
df["tag_count"] = df["tag_list"].str.len()
df["platform_count"] = df[["windows", "mac", "linux"]].sum(axis=1)
df["has_achievements"] = df["achievements_total"].fillna(0).gt(0)
df["has_controller"] = df["category_list"].apply(lambda items: any("controller" in item.lower() for item in items))
df["has_cloud"] = df["category_list"].apply(lambda items: any("steam cloud" in item.lower() for item in items))
df["has_family_sharing"] = df["category_list"].apply(lambda items: any("family sharing" in item.lower() for item in items))
df["is_single_player"] = df["category_list"].apply(lambda items: "Single-player" in items)
df["is_self_published"] = df.apply(
    lambda row: str(row["developers"]).strip().lower() == str(row["publishers"]).strip().lower(), axis=1
)

PRICE_ORDER = ["무료", "~$5", "$5~$10", "$10~$15", "$15~$20", "$20+", "가격 정보 없음"]
BAND_ORDER = ["무반응(리뷰 0개)", "미약 반응(리뷰 1~3개)", "제한 반응(리뷰 4~6개)", "반응 임계권(리뷰 7~9개)"]

df.head()

,appid,positive,negative,price,genres,total_reviews,name,developers,release_date,short_description,publishers,categories,windows,mac,linux,recommendations_total,achievements_total,owners_lower,owners_higher,tags,genre_list,tag_list,category_list,release_year,release_month,review_band,price_tier,positive_rate,description_len,genre_count,tag_count,platform_count,has_achievements,has_controller,has_cloud,has_family_sharing,is_single_player,is_self_published
0,370630,1,0,6.99,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strategy']",1,Sylia,Aldorlea Games,2025-04-30,"Pets vs Aliens! When your heroes are not good enough, all hope seems lost... not yet! Two faithful companions kickst...",Aldorlea Games,"Single-player, Steam Trading Cards, Steam Cloud, Family Sharing",True,False,False,NaN,NaN,20000,50000,"{""2D"": 361, ""Dog"": 461, ""RPG"": 450, ""CRPG"": 387, ""Cats"": 466, ""Cute"": 313, ""JRPG"": 411, ""Aliens"": 470, ""Casual"": 458...","[Adventure, Casual, Indie, RPG, Strategy]","[2D, Dog, RPG, CRPG, Cats, Cute, JRPG, Aliens, Casual, Sci-fi, Cartoon, Colorful, Strategy, Top-Down, Adventure, Emo...","[Single-player, Steam Trading Cards, Steam Cloud, Family Sharing]",2025,4,미약 반응(리뷰 1~3개),$5~$10,1.000000,261,5,20,1,False,False,True,True,True,True
1,400390,6,0,11.99,"['Casual', 'Indie']",6,Delusion,Silver Line Games,2023-05-26,"Delusion is an unusual puzzle game that tells a story about a beautiful world, full of life and colors, which change...",Silver Line Games,"Single-player, Steam Achievements, Full controller support, Steam Cloud, Stats, Remote Play on Phone, Remote Play on...",True,True,True,NaN,16.0,0,20000,"{""2.5D"": 84, ""Indie"": 45, ""Logic"": 57, ""Space"": 53, ""Casual"": 66, ""Puzzle"": 95, ""Mystery"": 48, ""Surreal"": 78, ""Color...","[Casual, Indie]","[2.5D, Indie, Logic, Space, Casual, Puzzle, Mystery, Surreal, Colorful, Relaxing, Stylized, Mythology, Narration, No...","[Single-player, Steam Achievements, Full controller support, Steam Cloud, Stats, Remote Play on Phone, Remote Play o...",2023,5,제한 반응(리뷰 4~6개),$10~$15,1.000000,192,2,20,3,True,True,True,True,True,True
2,520410,1,1,2.99,"['Action', 'Indie']",2,SpacePod,Robusta Gameworks,2023-05-04,Classic SpacePod now available on Steam!,Fabulous,"Single-player, Family Sharing",True,False,False,NaN,NaN,0,20000,"{""3D"": 33, ""Indie"": 20, ""Action"": 65, ""Combat"": 22, ""Sci-fi"": 30, ""Shooter"": 39, ""Multiplayer"": 20, ""Collectathon"": ...","[Action, Indie]","[3D, Indie, Action, Combat, Sci-fi, Shooter, Multiplayer, Collectathon, Singleplayer, Character Customization]","[Single-player, Family Sharing]",2023,5,미약 반응(리뷰 1~3개),~$5,0.500000,40,2,10,1,False,False,False,True,True,False
3,544690,2,1,0.99,"['Adventure', 'Casual', 'Indie']",3,Kriaturaz - O Guardião das Lendas (Base),Messier Games & Animations,2023-03-18,This is a Open Source Game. Kriaturaz is a game that presents the myths and the Brazilian legends in an unprecedente...,Messier Games & Animations,"Single-player, Family Sharing",True,False,False,NaN,NaN,0,20000,"{""3D"": 24, ""PvE"": 61, ""Cute"": 109, ""Indie"": 193, ""Casual"": 225, ""Combat"": 78, ""Fantasy"": 201, ""Colorful"": 123, ""Adve...","[Adventure, Casual, Indie]","[3D, PvE, Cute, Indie, Casual, Combat, Fantasy, Colorful, Adventure, Open World, Story Rich, Atmospheric, Collectath...","[Single-player, Family Sharing]",2023,3,미약 반응(리뷰 1~3개),~$5,0.666667,289,3,20,1,False,False,False,True,True,True
4,547670,8,0,9.99,"['Casual', 'Indie', 'Simulation']",8,Musical Range,Rockhopper Studios,2024-04-02,"Are you ready to play music for a crowd going crazy for you? ROCK, PLAY and PERFORM 25 great official songs in multi...",Rockhopper Studios,"Single-player, Tracked Controller Support, VR Only, Steam Leaderboards, Family Sharing",True,False,False,NaN,NaN,0,20000,"{""VR"": 14, ""Indie"": 32, ""Music"": 13, ""Casual"": 32, ""Rhythm"": 12, ""Shooter"": 12, ""Simulation"": 32}","[Casual, Indie, Simulation]","[VR, Indie, Music, Casual, Rhythm, Shooter, Simula

## 1. 실패 기준 정의와 전체 기준 비중

이 노트북에서는 `total_reviews < 10`을 “상업적 실패” 자체가 아니라, Steam에서 최소한의 초기 반응을 확보하지 못한 **무반응 게임**으로 정의한다.

비중을 볼 때는 `steam_indie_games_silence.csv` 내부만 보면 안 된다. 이 파일은 이미 리뷰 10개 미만 게임만 분리한 데이터이므로, 전체 시장 내 규모를 보려면 리뷰 10개 이상 게임인 `steam_indie_games.csv`와 합쳐서 비교해야 한다.

따라서 첫 분석은 전체 인디게임 표본에서 `리뷰 10개 미만`이 차지하는 비율을 확인하고, 그 다음에 무반응 그룹 내부의 리뷰 수 분포를 보조적으로 확인한다.

In [142]:
population_summary = (
    population_df.groupby("response_group")
    .agg(
        game_count=("appid", "nunique"),
        median_reviews=("total_reviews", "median"),
        avg_reviews=("total_reviews", "mean"),
    )
    .reset_index()
)
population_summary["ratio"] = population_summary["game_count"] / population_summary["game_count"].sum() * 100
population_summary["label"] = population_summary.apply(
    lambda row: f"{row['game_count']:,}개<br>{row['ratio']:.1f}%", axis=1
)

group_order = ["무반응(리뷰 10개 미만)", "반응 확보(리뷰 10개 이상)"]
population_summary["response_group"] = pd.Categorical(
    population_summary["response_group"], categories=group_order, ordered=True
)
population_summary = population_summary.sort_values("response_group")

display(population_summary.round(2))

fig = px.bar(
    population_summary,
    x="response_group",
    y="game_count",
    text="label",
    color="response_group",
    color_discrete_map={
        "무반응(리뷰 10개 미만)": COLOR_RISK,
        "반응 확보(리뷰 10개 이상)": COLOR_MAIN,
    },
    title="리뷰 10개 미만 게임 비중",
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_title="반응 그룹", yaxis_title="게임 수", showlegend=False)
fig.show()

basic_summary = pd.DataFrame(
    {
        "지표": [
            "전체 표본 게임 수",
            "리뷰 10개 미만 게임 수",
            "리뷰 10개 미만 비율",
            "무반응 그룹 리뷰 수 중앙값",
            "무반응 그룹 리뷰 0개 게임 수",
            "무반응 그룹 리뷰 7~9개 기준 근접 게임 수",
            "무반응 그룹 출시 연도 범위",
        ],
        "값": [
            f"{len(population_df):,}",
            f"{len(df):,}",
            f"{len(df) / len(population_df) * 100:.1f}%",
            f"{df['total_reviews'].median():.0f}",
            f"{(df['total_reviews'] == 0).sum():,}",
            f"{df['total_reviews'].between(7, 9).sum():,}",
            f"{int(df['release_year'].min())}~{int(df['release_year'].max())}",
        ],
    }
)
display(basic_summary)

review_dist = df["total_reviews"].value_counts().sort_index().reset_index()
review_dist.columns = ["total_reviews", "game_count"]
review_dist["ratio"] = review_dist["game_count"] / review_dist["game_count"].sum() * 100

fig = px.bar(
    review_dist,
    x="total_reviews",
    y="game_count",
    text=review_dist["ratio"].map(lambda x: f"{x:.1f}%"),
    color_discrete_sequence=[COLOR_MAIN],
    title="리뷰 10개 미만 게임 내부의 리뷰 수 분포",
)
fig.update_layout(xaxis_title="총 리뷰 수", yaxis_title="게임 수", showlegend=False)
fig.show()

,response_group,game_count,median_reviews,avg_reviews,ratio,label
0,무반응(리뷰 10개 미만),6676,3.0,3.78,43.33,"6,676개<br>43.3%"
1,반응 확보(리뷰 10개 이상),8730,40.0,774.10,56.67,"8,730개<br>56.7%"


,지표,값
0,전체 표본 게임 수,"15,406"
1,리뷰 10개 미만 게임 수,"6,676"
2,리뷰 10개 미만 비율,43.3%
3,무반응 그룹 리뷰 수 중앙값,3
4,무반응 그룹 리뷰 0개 게임 수,138
5,무반응 그룹 리뷰 7~9개 기준 근접 게임 수,"1,155"
6,무반응 그룹 출시 연도 범위,2023~2025


**해석:** 이 차트는 실패 게임 내부에서도 상태가 다르다는 점을 보여준다. 리뷰 0개는 무반응이고, 7~9개는 기준에 근접한 경계 그룹이다. PPT에서는 “실패 게임”을 하나로 묶기보다 유저 반응 정도를 나누어 설명하면 원인 분석이 더 설득력 있다.

## 2. 유저 반응 정도별 그룹화

리뷰 수 0~9개를 네 단계로 나누면, 무반응인지 아니면 소수 유저에게는 도달했지만 확산되지 못했는지를 구분할 수 있다.

In [143]:
band_summary = (
    df.groupby("review_band", observed=True)
    .agg(
        game_count=("appid", "count"),
        median_price=("price", "median"),
        median_positive_rate=("positive_rate", "median"),
        median_description_len=("description_len", "median"),
        median_tag_count=("tag_count", "median"),
        achievement_rate=("has_achievements", "mean"),
        controller_rate=("has_controller", "mean"),
    )
    .reindex(BAND_ORDER)
    .reset_index()
)
band_summary["game_ratio"] = band_summary["game_count"] / band_summary["game_count"].sum() * 100
for col in ["achievement_rate", "controller_rate"]:
    band_summary[col] = band_summary[col] * 100

display(band_summary.round(2))

fig = px.bar(
    band_summary,
    x="review_band",
    y="game_count",
    text=band_summary["game_ratio"].map(lambda x: f"{x:.1f}%"),
    color="review_band",
    color_discrete_sequence=[COLOR_RISK, COLOR_ACCENT, COLOR_MAIN, COLOR_GOOD],
    title="유저 반응 정도별 게임 수",
)
fig.update_layout(xaxis_title="반응 정도", yaxis_title="게임 수", showlegend=False)
fig.show()

,review_band,game_count,median_price,median_positive_rate,median_description_len,median_tag_count,achievement_rate,controller_rate,game_ratio
0,무반응(리뷰 0개),138,2.00,NaN,186.5,20.0,70.29,28.99,2.07
1,미약 반응(리뷰 1~3개),3400,3.99,1.00,207.0,18.0,47.06,33.09,50.93
2,제한 반응(리뷰 4~6개),1983,3.99,1.00,209.0,19.0,59.35,36.26,29.70
3,반응 임계권(리뷰 7~9개),1155,4.99,0.88,213.0,20.0,61.39,37.75,17.30


**해석:** 무반응 그룹은 노출 또는 구매 전환 자체가 거의 일어나지 않은 게임으로 볼 수 있다. 반면 반응 임계권 그룹은 최소한의 유저 반응은 있었지만, Steam 평가 기준이나 사회적 증거로 확장되기 전에 멈춘 게임이다. 이후 분석에서는 이 차이를 염두에 두고 장르·가격·신뢰 신호를 확인한다.

## 3. 출시 시점 분석

출시 시점은 게임 품질과 별개로 초기 노출 경쟁에 영향을 준다. 연도별·월별 분포를 보면 무반응 게임이 특정 시기 또는 최근 출시 과잉과 연결되는지 확인할 수 있다.

In [144]:
yearly = (
    df.dropna(subset=["release_year"])
    .groupby("release_year")
    .agg(
        game_count=("appid", "count"),
        median_reviews=("total_reviews", "median"),
        zero_review_rate=("total_reviews", lambda s: (s == 0).mean() * 100),
        near_threshold_rate=("total_reviews", lambda s: s.between(7, 9).mean() * 100),
    )
    .reset_index()
)
yearly["release_year"] = yearly["release_year"].astype(int)

display(yearly.tail(10).round(2))

fig = go.Figure()
fig.add_trace(go.Bar(x=yearly["release_year"], y=yearly["game_count"], name="무반응 게임 수", marker_color=COLOR_MAIN))
fig.add_trace(
    go.Scatter(
        x=yearly["release_year"],
        y=yearly["zero_review_rate"],
        name="리뷰 0개 비율",
        mode="lines+markers",
        yaxis="y2",
        line=dict(color=COLOR_RISK),
    )
)
fig.update_layout(
    title="연도별 무반응 게임 수와 완전 침묵 비율",
    xaxis_title="출시 연도",
    yaxis=dict(title="게임 수"),
    yaxis2=dict(title="리뷰 0개 비율(%)", overlaying="y", side="right"),
    legend=dict(orientation="h", y=1.08),
)
fig.show()

monthly = (
    df.dropna(subset=["release_month"])
    .groupby("release_month")
    .agg(game_count=("appid", "count"), median_reviews=("total_reviews", "median"))
    .reset_index()
)
monthly["release_month"] = monthly["release_month"].astype(int)

fig = px.bar(
    monthly,
    x="release_month",
    y="game_count",
    color_discrete_sequence=[COLOR_ACCENT],
    title="월별 무반응 게임 출시 분포",
)
fig.update_layout(xaxis_title="출시 월", yaxis_title="게임 수", xaxis=dict(dtick=1))
fig.show()

,release_year,game_count,median_reviews,zero_review_rate,near_threshold_rate
0,2023,2065,3.0,1.11,18.89
1,2024,3058,3.0,1.93,16.45
2,2025,1553,3.0,3.61,16.87


**해석:** 연도별 그래프는 PPT에서 “시장 과밀로 인해 출시 후 묻히는 게임이 누적되고 있다”는 문제 제기용으로 적합하다. 월별 분포는 특정 출시 시즌에 무반응 게임이 몰리는지 확인하는 보조 근거로 사용한다.

## 4. 장르 분석

실패 게임의 장르는 단순 빈도와 침묵 강도를 함께 봐야 한다. 게임 수가 많은 장르는 당연히 많이 등장하므로, 장르별 리뷰 수 중앙값과 무반응 비율을 같이 확인한다.

In [145]:
genre_df = df[["appid", "genre_list", "total_reviews", "review_band", "price", "positive_rate"]].explode("genre_list")
genre_df = genre_df.rename(columns={"genre_list": "genre"}).dropna(subset=["genre"])
genre_summary = (
    genre_df.groupby("genre")
    .agg(
        game_count=("appid", "nunique"),
        median_reviews=("total_reviews", "median"),
        zero_review_rate=("total_reviews", lambda s: (s == 0).mean() * 100),
        near_threshold_rate=("total_reviews", lambda s: s.between(7, 9).mean() * 100),
        median_price=("price", "median"),
    )
    .query("game_count >= 50")
    .query("genre != 'Indie'")
    .sort_values(["game_count", "zero_review_rate"], ascending=[False, False])
    .reset_index()
)

display(genre_summary.round(2))

genre_plot = genre_summary.sort_values("game_count", ascending=False)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=genre_plot["genre"],
        y=genre_plot["game_count"],
        name="무반응 게임 수",
        marker_color=COLOR_MAIN,
        text=genre_plot["game_count"].map(lambda x: f"{x:,}"),
        textposition="outside",
    )
)
fig.add_trace(
    go.Scatter(
        x=genre_plot["genre"],
        y=genre_plot["zero_review_rate"],
        name="리뷰 0개 비율",
        mode="lines+markers+text",
        yaxis="y2",
        line=dict(color=COLOR_RISK, width=3),
        marker=dict(size=8),
        text=genre_plot["zero_review_rate"].map(lambda x: f"{x:.1f}%"),
        textposition="top center",
    )
)
fig.update_layout(
    title="장르별 무반응 게임 수와 리뷰 0개 비율",
    xaxis_title="장르",
    yaxis=dict(title="무반응 게임 수", rangemode="tozero"),
    yaxis2=dict(title="리뷰 0개 비율(%)", overlaying="y", side="right", rangemode="tozero"),
    legend=dict(orientation="h", y=1.12),
    margin=dict(t=90),
)
fig.show()

fig = px.bar(
    genre_summary.sort_values("game_count", ascending=True).tail(15),
    x="game_count",
    y="genre",
    orientation="h",
    color="zero_review_rate",
    color_continuous_scale="Reds",
    title="무반응 게임이 많이 관측되는 주요 장르 Top 15",
)
fig.update_layout(xaxis_title="무반응 게임 수", yaxis_title="장르")
fig.show()

,genre,game_count,median_reviews,zero_review_rate,near_threshold_rate,median_price
0,Casual,3589,3.0,3.15,16.38,3.19
1,Action,3049,3.0,1.34,17.45,3.99
2,Adventure,2865,3.0,1.33,18.60,3.99
3,Strategy,1411,3.0,1.77,16.37,3.99
4,Simulation,1236,4.0,2.02,20.55,3.99
5,RPG,1134,3.0,1.32,19.40,4.99
6,Racing,274,3.0,4.01,17.88,3.99
7,Sports,249,3.0,3.61,22.09,4.99


**해석:** 장르 분석은 “어떤 장르가 나쁘다”가 아니라 “어떤 장르에서 무반응 위험이 더 자주 관측되는가”를 보여주는 용도다. 특히 Casual, Adventure, Simulation처럼 진입 장벽이 낮고 공급이 많은 장르는 차별화 신호가 약하면 Steam에서 묻힐 가능성이 높다는 식으로 해석한다.

## 5. 가격대 분석

가격은 유저의 기대치를 만든다. 무명 인디게임은 가격이 높으면 구매 전 설득 부담이 커지고, 너무 낮으면 품질 신뢰 신호가 약해질 수 있다.

In [146]:
price_summary = (
    df.groupby("price_tier", observed=True)
    .agg(
        game_count=("appid", "count"),
        median_reviews=("total_reviews", "median"),
        avg_reviews=("total_reviews", "mean"),
        zero_review_rate=("total_reviews", lambda s: (s == 0).mean() * 100),
        near_threshold_rate=("total_reviews", lambda s: s.between(7, 9).mean() * 100),
        achievement_rate=("has_achievements", "mean"),
        median_description_len=("description_len", "median"),
    )
    .reindex(PRICE_ORDER)
    .dropna(subset=["game_count"])
    .reset_index()
)
price_summary["achievement_rate"] *= 100

display(price_summary.round(2))

fig = go.Figure()
fig.add_trace(go.Bar(x=price_summary["price_tier"], y=price_summary["game_count"], name="게임 수", marker_color=COLOR_MAIN))
fig.add_trace(
    go.Scatter(
        x=price_summary["price_tier"],
        y=price_summary["zero_review_rate"],
        name="리뷰 0개 비율",
        mode="lines+markers",
        yaxis="y2",
        line=dict(color=COLOR_RISK),
    )
)
fig.update_layout(
    title="가격대별 무반응 게임 수와 완전 침묵 비율",
    xaxis_title="가격대",
    yaxis=dict(title="게임 수"),
    yaxis2=dict(title="리뷰 0개 비율(%)", overlaying="y", side="right"),
    legend=dict(orientation="h", y=1.08),
)
fig.show()

fig = px.box(
    df,
    x="price_tier",
    y="total_reviews",
    category_orders={"price_tier": PRICE_ORDER},
    color="price_tier",
    title="가격대별 리뷰 수 분포: 모두 10개 미만인 그룹 내부 비교",
)
fig.update_layout(xaxis_title="가격대", yaxis_title="총 리뷰 수", showlegend=False)
fig.show()

,price_tier,game_count,median_reviews,avg_reviews,zero_review_rate,near_threshold_rate,achievement_rate,median_description_len
0,~$5,4676.0,3.0,3.73,2.14,16.34,52.93,203.0
1,$5~$10,1423.0,3.0,3.93,1.90,19.54,56.08,218.0
2,$10~$15,347.0,4.0,4.06,2.02,22.48,58.79,226.0
3,$15~$20,132.0,4.0,3.92,1.52,15.91,59.09,246.0
4,$20+,98.0,2.5,3.27,2.04,14.29,28.57,224.5


**해석:** 가격대별 비교는 “적정 가격”을 직접 결론내기보다, 가격이 유저의 기대치와 신뢰 판단에 어떤 부담을 주는지 보여준다. 같은 무반응 그룹 안에서도 특정 가격대의 리뷰 0개 비율이 높다면, 해당 가격대에서는 상점 페이지 설득력이나 장르 기대치 관리가 더 중요하다고 볼 수 있다.

## 6. 태그 분석

태그는 유저가 게임을 발견하고 기대를 형성하는 핵심 단서다. 실패 게임에서 자주 반복되는 태그는 공급 과잉 또는 차별화 부족 가능성을 보여준다.

In [147]:
tag_df = df[["appid", "tag_list", "total_reviews", "review_band", "price", "genre_count"]].explode("tag_list")
tag_df = tag_df.rename(columns={"tag_list": "tag"}).dropna(subset=["tag"])

tag_summary = (
    tag_df.groupby("tag")
    .agg(
        game_count=("appid", "nunique"),
        median_reviews=("total_reviews", "median"),
        zero_review_rate=("total_reviews", lambda s: (s == 0).mean() * 100),
        near_threshold_rate=("total_reviews", lambda s: s.between(7, 9).mean() * 100),
        median_price=("price", "median"),
    )
    .query("game_count >= 100")
    .query("tag != 'Indie'")
    .sort_values("game_count", ascending=False)
    .reset_index()
)

display(tag_summary.head(30).round(2))

top_tags = tag_summary.head(20).sort_values("game_count", ascending=True)
fig = px.bar(
    top_tags,
    x="game_count",
    y="tag",
    orientation="h",
    color="zero_review_rate",
    color_continuous_scale="Reds",
    title="무반응 게임에서 가장 자주 등장하는 태그 Top 20",
)
fig.update_layout(
    xaxis_title="무반응 게임 수",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=top_tags["tag"].tolist(),
        ticktext=top_tags["tag"].tolist(),
        automargin=True,
    ),
    height=720,
    margin=dict(l=190, t=90, r=120, b=80),
    coloraxis_colorbar_title="리뷰 0개 비율",
)
fig.show()

high_zero_tags = tag_summary.sort_values("zero_review_rate", ascending=False).head(20)
high_zero_plot = high_zero_tags.sort_values("zero_review_rate", ascending=True)
fig = px.bar(
    high_zero_plot,
    x="zero_review_rate",
    y="tag",
    orientation="h",
    color="game_count",
    color_continuous_scale="Oranges",
    title="리뷰 0개 비율이 높은 주요 태그 Top 20",
)
fig.update_layout(
    xaxis_title="리뷰 0개 비율(%)",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=high_zero_plot["tag"].tolist(),
        ticktext=high_zero_plot["tag"].tolist(),
        automargin=True,
    ),
    height=720,
    margin=dict(l=190, t=90, r=120, b=80),
    coloraxis_colorbar_title="무반응 게임 수",
)
fig.show()

,tag,game_count,median_reviews,zero_review_rate,near_threshold_rate,median_price
0,Singleplayer,5290,3.0,2.12,16.99,3.99
1,Casual,3515,3.0,2.93,16.30,2.99
2,2D,3110,3.0,2.67,16.21,3.99
3,Action,3005,3.0,1.40,17.17,3.99
4,Adventure,2709,3.0,1.40,18.46,3.99
5,3D,2395,3.0,1.25,17.04,3.99
6,Colorful,1807,3.0,1.99,16.93,3.99
7,Puzzle,1638,3.0,3.42,15.93,3.99
8,Pixel Graphics,1625,3.0,0.98,17.85,3.99
9,Atmospheric,1458,4.0,1.37,19.75,3.99


### 6-1. 전체 대비 무반응 태그 과대표현 분석

앞의 Top 20 그래프는 무반응 게임 내부에서 많이 등장하는 태그를 보여준다. 하지만 `Singleplayer`, `Casual`, `Action` 같은 태그는 전체 게임에서도 흔할 수 있으므로, 실패 신호로 해석하려면 **전체 표본 내 태그 비중**과 **무반응 게임 내 태그 비중**을 비교해야 한다.

아래 분석에서는 `무반응 태그 비중 - 전체 태그 비중`을 계산해, 무반응 게임에서 상대적으로 더 많이 등장하는 태그를 확인한다.

In [148]:
response_tag_df = df_response[["appid", "tags"]].copy()
response_tag_df["tag_list"] = response_tag_df["tags"].apply(parse_tag_names)

silence_tag_presence = (
    df[["appid", "tag_list"]]
    .explode("tag_list")
    .rename(columns={"tag_list": "tag"})
    .dropna(subset=["tag"])
    .query("tag != 'Indie'")
    .drop_duplicates(["appid", "tag"])
)

response_tag_presence = (
    response_tag_df[["appid", "tag_list"]]
    .explode("tag_list")
    .rename(columns={"tag_list": "tag"})
    .dropna(subset=["tag"])
    .query("tag != 'Indie'")
    .drop_duplicates(["appid", "tag"])
)

population_tag_presence = pd.concat(
    [
        silence_tag_presence.assign(response_group="무반응"),
        response_tag_presence.assign(response_group="반응 확보"),
    ],
    ignore_index=True,
).drop_duplicates(["appid", "tag"], keep="first")

silence_game_count = df["appid"].nunique()
population_game_count = population_df["appid"].nunique()

silence_tag_share = (
    silence_tag_presence.groupby("tag")["appid"]
    .nunique()
    .rename("silence_game_count")
    .reset_index()
)
silence_tag_share["silence_share"] = silence_tag_share["silence_game_count"] / silence_game_count * 100

population_tag_share = (
    population_tag_presence.groupby("tag")["appid"]
    .nunique()
    .rename("population_game_count")
    .reset_index()
)
population_tag_share["population_share"] = population_tag_share["population_game_count"] / population_game_count * 100

tag_lift = (
    silence_tag_share.merge(population_tag_share, on="tag", how="left")
    .fillna({"population_game_count": 0, "population_share": 0})
)
tag_lift["share_gap"] = tag_lift["silence_share"] - tag_lift["population_share"]
tag_lift["share_ratio"] = np.where(
    tag_lift["population_share"] > 0,
    tag_lift["silence_share"] / tag_lift["population_share"],
    np.nan,
)

tag_lift = tag_lift.query("silence_game_count >= 100").sort_values("share_gap", ascending=False)

display(tag_lift.head(30).round(2))

plot_tags = tag_lift.head(12).sort_values("share_gap", ascending=True)
share_compare_df = plot_tags.melt(
    id_vars=["tag", "share_gap"],
    value_vars=["population_share", "silence_share"],
    var_name="group",
    value_name="share",
)
share_compare_df["group"] = share_compare_df["group"].map(
    {
        "population_share": "전체 게임",
        "silence_share": "무반응 게임",
    }
)
share_compare_df["label"] = share_compare_df["share"].map(lambda x: f"{x:.1f}%")

fig = px.bar(
    share_compare_df,
    x="share",
    y="tag",
    color="group",
    barmode="group",
    orientation="h",
    text="label",
    category_orders={
        "tag": plot_tags["tag"].tolist(),
        "group": ["전체 게임", "무반응 게임"],
    },
    color_discrete_map={
        "전체 게임": COLOR_MUTED,
        "무반응 게임": COLOR_RISK,
    },
    title="전체 게임 vs 무반응 게임의 태그 비중 비교",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="해당 태그를 가진 게임 비중(%)",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=plot_tags["tag"].tolist(),
        ticktext=plot_tags["tag"].tolist(),
        automargin=True,
    ),
    height=620,
    margin=dict(l=190, t=90, r=150, b=80),
    legend=dict(orientation="h", y=1.08),
)
fig.show()

gap_labels = plot_tags.copy()
gap_labels["gap_label"] = gap_labels["share_gap"].map(lambda x: f"+{x:.1f}%p")
fig = px.bar(
    gap_labels,
    x="share_gap",
    y="tag",
    orientation="h",
    text="gap_label",
    color_discrete_sequence=[COLOR_RISK],
    title="무반응 게임에서 상대적으로 더 많이 나타나는 태그",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="무반응 비중 - 전체 비중 (%p)",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=gap_labels["tag"].tolist(),
        ticktext=gap_labels["tag"].tolist(),
        automargin=True,
    ),
    height=620,
    margin=dict(l=190, t=90, r=150, b=80),
    showlegend=False,
)
fig.show()

,tag,silence_game_count,silence_share,population_game_count,population_share,share_gap,share_ratio
329,Singleplayer,5290,79.24,9865,64.03,15.21,1.24
69,Casual,3515,52.65,5975,38.78,13.87,1.36
3,2D,3110,46.58,5691,36.94,9.64,1.26
7,3D,2395,35.87,4258,27.64,8.24,1.30
17,Action,3005,45.01,5740,37.26,7.75,1.21
31,Arcade,1448,21.69,2361,15.33,6.36,1.42
86,Colorful,1807,27.07,3393,22.02,5.04,1.23
286,Puzzle,1638,24.54,3076,19.97,4.57,1.23
23,Adventure,2709,40.58,5569,36.15,4.43,1.12
272,Pixel Graphics,1625,24.34,3148,20.43,3.91,1.19


**해석:** 이 그래프는 각 태그가 전체 게임과 무반응 게임에서 각각 얼마나 자주 등장하는지 나란히 보여준다. 빨간 막대가 회색 막대보다 길수록, 해당 태그는 전체 평균보다 무반응 게임에 더 많이 몰려 있다는 뜻이다.

따라서 `Singleplayer`, `Casual`, `2D`, `3D`, `Action`처럼 차이가 큰 태그는 “무반응 게임에서 상대적으로 더 많이 관측되는 시장 언어”로 해석할 수 있다. 이 태그들이 실패 원인이라는 뜻은 아니며, 범용 태그만으로는 유저에게 구체적인 선택 이유를 만들기 어렵다는 신호로 보는 것이 안전하다.

### 6-2. 리뷰 10개 이상 게임에서 과대표현된 태그

무반응 그룹과 반대로, 리뷰 10개 이상을 확보한 게임에서 전체 표본 대비 더 많이 나타나는 태그도 확인한다. 이 비교는 “반응을 얻은 게임은 어떤 플레이 경험을 더 구체적으로 제시했는가”를 보는 용도다.

In [149]:
response_tag_share = (
    response_tag_presence.groupby("tag")["appid"]
    .nunique()
    .rename("response_game_count")
    .reset_index()
)
response_game_count = df_response["appid"].nunique()
response_tag_share["response_share"] = response_tag_share["response_game_count"] / response_game_count * 100

response_tag_lift = response_tag_share.merge(population_tag_share, on="tag", how="left")
response_tag_lift["share_gap"] = response_tag_lift["response_share"] - response_tag_lift["population_share"]
response_tag_lift["share_ratio"] = np.where(
    response_tag_lift["population_share"] > 0,
    response_tag_lift["response_share"] / response_tag_lift["population_share"],
    np.nan,
)
response_tag_lift = (
    response_tag_lift.query("response_game_count >= 100")
    .sort_values("share_gap", ascending=False)
    .reset_index(drop=True)
)

display(response_tag_lift.head(30).round(2))

response_plot_tags = response_tag_lift.head(12).sort_values("share_gap", ascending=True)
response_share_compare_df = response_plot_tags.melt(
    id_vars=["tag", "share_gap"],
    value_vars=["population_share", "response_share"],
    var_name="group",
    value_name="share",
)
response_share_compare_df["group"] = response_share_compare_df["group"].map(
    {
        "population_share": "전체 게임",
        "response_share": "리뷰 10개 이상 게임",
    }
)
response_share_compare_df["label"] = response_share_compare_df["share"].map(lambda x: f"{x:.1f}%")

fig = px.bar(
    response_share_compare_df,
    x="share",
    y="tag",
    color="group",
    barmode="group",
    orientation="h",
    text="label",
    category_orders={
        "tag": response_plot_tags["tag"].tolist(),
        "group": ["전체 게임", "리뷰 10개 이상 게임"],
    },
    color_discrete_map={
        "전체 게임": COLOR_MUTED,
        "리뷰 10개 이상 게임": COLOR_GOOD,
    },
    title="전체 게임 vs 리뷰 10개 이상 게임의 태그 비중 비교",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="해당 태그를 가진 게임 비중(%)",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=response_plot_tags["tag"].tolist(),
        ticktext=response_plot_tags["tag"].tolist(),
        automargin=True,
    ),
    height=620,
    margin=dict(l=190, t=90, r=150, b=80),
    legend=dict(orientation="h", y=1.08),
)
fig.show()

response_gap_labels = response_plot_tags.copy()
response_gap_labels["gap_label"] = response_gap_labels["share_gap"].map(lambda x: f"+{x:.1f}%p")
fig = px.bar(
    response_gap_labels,
    x="share_gap",
    y="tag",
    orientation="h",
    text="gap_label",
    color_discrete_sequence=[COLOR_GOOD],
    title="리뷰 10개 이상 게임에서 상대적으로 더 많이 나타나는 태그",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="리뷰 10개 이상 비중 - 전체 비중 (%p)",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=response_gap_labels["tag"].tolist(),
        ticktext=response_gap_labels["tag"].tolist(),
        automargin=True,
    ),
    height=620,
    margin=dict(l=190, t=90, r=150, b=80),
    showlegend=False,
)
fig.show()

,tag,response_game_count,response_share,population_game_count,population_share,share_gap,share_ratio
0,Story Rich,1542,17.66,2516,16.33,1.33,1.08
1,Rogue-lite,849,9.73,1360,8.83,0.90,1.10
2,Building,531,6.08,806,5.23,0.85,1.16
3,Early Access,425,4.87,630,4.09,0.78,1.19
4,Sandbox,548,6.28,848,5.50,0.77,1.14
5,Co-op,461,5.28,699,4.54,0.74,1.16
6,Online Co-Op,342,3.92,497,3.23,0.69,1.21
7,Management,519,5.95,815,5.29,0.65,1.12
8,Replay Value,221,2.53,298,1.93,0.60,1.31
9,Base-Building,358,4.10,544,3.53,0.57,1.16


**해석:** 리뷰 10개 이상을 확보한 게임은 전체 평균보다 `Story Rich`, `Rogue-lite`, `Building`, `Sandbox`, `Co-op`, `Management`, `Replay Value`처럼 플레이 경험이 더 구체적으로 상상되는 태그가 상대적으로 많이 나타난다.

무반응 게임이 `Singleplayer`, `Casual`, `2D`, `Action` 같은 넓은 범용 태그에 더 치우쳐 있었다면, 반응 확보 게임은 서사, 반복 플레이, 시스템 운영, 협동 플레이처럼 유저가 기대할 수 있는 플레이 루프를 더 명확하게 제시하는 경향이 있다.

PPT에서는 “반응을 얻은 게임은 단순 장르 태그보다 구체적 경험 태그가 더 두드러졌다”는 메시지로 연결할 수 있다.

### 6-3. 태그별 반응 확보 유의성 검정

리뷰 10개 이상 게임에서 많이 보이는 태그가 단순 빈도 차이인지, 전체 표본 기준으로도 의미 있는 차이인지 확인한다. 각 태그에 대해 `리뷰 10개 이상 여부 × 태그 보유 여부` 2x2 표를 만들고, 기대빈도가 낮으면 Fisher exact test, 충분하면 카이제곱 검정을 적용한다. 여러 태그를 동시에 검정하므로 Benjamini-Hochberg FDR 보정 q-value를 함께 본다.

In [150]:
def build_population_feature_df(silence_df, response_df):
    silence_features = silence_df.copy()
    response_features = response_df.copy()
    all_games = pd.concat(
        [
            silence_features.assign(response_group="무반응(리뷰 10개 미만)"),
            response_features.assign(response_group="반응 확보(리뷰 10개 이상)"),
        ],
        ignore_index=True,
    ).drop_duplicates("appid", keep="first")

    for col in ["windows", "mac", "linux"]:
        all_games[col] = all_games[col].apply(to_bool)

    all_games["genre_list"] = all_games["genres"].apply(parse_list_literal)
    all_games["tag_list"] = all_games["tags"].apply(parse_tag_names)
    all_games["category_list"] = all_games["categories"].apply(split_categories)
    all_games["release_date"] = pd.to_datetime(all_games["release_date"], errors="coerce")
    all_games["release_year"] = all_games["release_date"].dt.year
    all_games["price_tier"] = all_games["price"].apply(make_price_tier)
    all_games["positive_rate"] = np.where(
        all_games["total_reviews"] > 0,
        all_games["positive"] / all_games["total_reviews"] * 100,
        np.nan,
    )
    all_games["description_len"] = all_games["short_description"].fillna("").str.len()
    all_games["tag_count"] = all_games["tag_list"].str.len()
    all_games["platform_count"] = all_games[["windows", "mac", "linux"]].sum(axis=1)
    all_games["has_achievements"] = all_games["achievements_total"].fillna(0).gt(0)
    all_games["has_controller"] = all_games["category_list"].apply(lambda items: any("controller" in item.lower() for item in items))
    all_games["has_cloud"] = all_games["category_list"].apply(lambda items: any("steam cloud" in item.lower() for item in items))
    all_games["is_single_player"] = all_games["category_list"].apply(lambda items: "Single-player" in items)
    all_games["is_self_published"] = all_games.apply(
        lambda row: str(row["developers"]).strip().lower() == str(row["publishers"]).strip().lower(), axis=1
    )
    all_games["is_response"] = all_games["total_reviews"] >= 10
    return all_games


def test_binary_presence(presence_df, entity_col, games_df, min_count=50):
    base = games_df[["appid", "is_response", "total_reviews", "positive_rate"]].drop_duplicates("appid")
    total_games = base["appid"].nunique()
    total_response = int(base["is_response"].sum())
    overall_response_rate = total_response / total_games * 100

    presence = presence_df[["appid", entity_col]].dropna().drop_duplicates(["appid", entity_col])
    presence = presence.merge(base, on="appid", how="inner")

    rows = []
    for entity, group in presence.groupby(entity_col):
        game_count = group["appid"].nunique()
        if game_count < min_count:
            continue

        response_count = int(group["is_response"].sum())
        silence_count = game_count - response_count
        without_count = total_games - game_count
        without_response = total_response - response_count
        without_silence = without_count - without_response
        table = np.array([[response_count, silence_count], [without_response, without_silence]])

        _, chi_p, _, expected = chi2_contingency(table, correction=False)
        if (expected < 5).any():
            test_name = "Fisher exact"
            _, p_value = fisher_exact(table)
        else:
            test_name = "Chi-square"
            p_value = chi_p

        response_rate = response_count / game_count * 100
        without_response_rate = without_response / without_count * 100 if without_count else np.nan
        odds_ratio = ((response_count + 0.5) * (without_silence + 0.5)) / ((silence_count + 0.5) * (without_response + 0.5))

        rows.append(
            {
                entity_col: entity,
                "game_count": game_count,
                "response_count": response_count,
                "silence_count": silence_count,
                "response_rate": response_rate,
                "without_response_rate": without_response_rate,
                "diff_pp": response_rate - without_response_rate,
                "lift_vs_overall": response_rate / overall_response_rate,
                "odds_ratio": odds_ratio,
                "median_reviews": group["total_reviews"].median(),
                "median_positive_rate": group["positive_rate"].median(),
                "p_value": p_value,
                "test": test_name,
            }
        )

    result = pd.DataFrame(rows)
    if result.empty:
        return result
    result["q_value"] = multipletests(result["p_value"], method="fdr_bh")[1]
    return result.sort_values(["q_value", "diff_pp"], ascending=[True, False]).reset_index(drop=True)


all_games = build_population_feature_df(df, df_response)
all_tag_presence = (
    all_games[["appid", "tag_list"]]
    .explode("tag_list")
    .rename(columns={"tag_list": "tag"})
    .dropna(subset=["tag"])
    .query("tag != 'Indie'")
    .drop_duplicates(["appid", "tag"])
)

tag_test = test_binary_presence(all_tag_presence, "tag", all_games, min_count=100)
significant_response_tags = (
    tag_test
    .query("q_value < 0.05 and diff_pp > 0 and response_count >= 50")
    .sort_values(["diff_pp", "lift_vs_overall"], ascending=False)
    .reset_index(drop=True)
)

display(
    significant_response_tags[
        [
            "tag", "game_count", "response_rate", "without_response_rate", "diff_pp",
            "lift_vs_overall", "odds_ratio", "median_reviews", "median_positive_rate", "q_value", "test",
        ]
    ].head(30).round(3)
)

tag_test_plot = significant_response_tags.head(15).sort_values("diff_pp", ascending=True).copy()
tag_test_plot["label"] = tag_test_plot.apply(lambda row: f"+{row['diff_pp']:.1f}%p / q={row['q_value']:.3f}", axis=1)

fig = px.bar(
    tag_test_plot,
    x="diff_pp",
    y="tag",
    orientation="h",
    text="label",
    color="lift_vs_overall",
    color_continuous_scale="Greens",
    title="리뷰 10개 이상 도달률이 유의하게 높은 단일 태그",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="태그 보유 게임 반응률 - 미보유 게임 반응률 (%p)",
    yaxis_title="태그",
    yaxis=dict(
        tickmode="array",
        tickvals=tag_test_plot["tag"].tolist(),
        ticktext=tag_test_plot["tag"].tolist(),
        automargin=True,
    ),
    height=680,
    margin=dict(l=190, t=90, r=190, b=80),
    coloraxis_colorbar_title="전체 대비 lift",
)
fig.show()

,tag,game_count,response_rate,without_response_rate,diff_pp,lift_vs_overall,odds_ratio,median_reviews,median_positive_rate,q_value,test
0,Roguelike Deckbuilder,160,80.000,56.421,23.579,1.412,3.054,96.0,87.179,0.000,Chi-square
1,Turn-Based,173,79.769,56.404,23.365,1.408,3.016,96.0,89.328,0.000,Chi-square
2,Narrative,135,77.778,56.480,21.298,1.373,2.665,68.0,92.202,0.000,Chi-square
3,Colony Sim,185,76.216,56.429,19.788,1.345,2.455,75.0,80.919,0.000,Chi-square
4,Replay Value,298,74.161,56.321,17.840,1.309,2.217,49.5,88.235,0.000,Chi-square
5,Soundtrack,167,74.251,56.474,17.778,1.310,2.206,51.0,90.518,0.000,Chi-square
6,Online Co-Op,497,68.813,56.261,12.552,1.214,1.712,52.0,83.333,0.000,Chi-square
7,Dating Sim,215,68.372,56.501,11.872,1.207,1.658,33.0,90.476,0.001,Chi-square
8,Metroidvania,317,68.139,56.425,11.714,1.202,1.647,24.0,88.889,0.000,Chi-square
9,LGBTQ+,252,67.857,56.480,11.377,1.197,1.621,25.5,95.212,0.001,Chi-square


**해석:** 단일 태그 검정은 “반응 확보 게임에서 더 자주 관측되는 태그”를 통계적으로 걸러내는 단계다. 다만 태그는 서로 독립적이지 않으므로, 이 결과는 인과가 아니라 반응 확보와의 연관성으로 해석한다. 발표에서는 q-value와 함께 `diff_pp`, `lift_vs_overall`, 리뷰 수 중앙값을 같이 보여주는 것이 안전하다.

### 6-4. 반응을 이끌어낸 태그 조합 분석

유저는 단일 태그보다 태그 조합을 통해 게임의 플레이 경험을 상상한다. 여기서는 리뷰 10개 이상 게임에서 유의하게 높은 단일 태그 후보를 바탕으로 2개/3개 조합을 만들고, 각 조합의 반응률이 전체 평균과 각 단일 태그 대비로도 높은지 확인한다.

In [151]:
def build_combo_presence(games_df, candidate_tags, combo_size):
    candidate_set = set(candidate_tags)
    rows = []
    for appid, tags in games_df[["appid", "tag_list"]].itertuples(index=False):
        selected = sorted(set(tags) & candidate_set)
        if len(selected) < combo_size:
            continue
        for combo in combinations(selected, combo_size):
            rows.append({"appid": appid, "tag_combo": " + ".join(combo), "combo_size": combo_size})
    return pd.DataFrame(rows)


candidate_tags = significant_response_tags.head(40)["tag"].tolist()
single_tag_response_rate = tag_test.set_index("tag")["response_rate"].to_dict()

pair_presence = build_combo_presence(all_games, candidate_tags, combo_size=2)
triple_presence = build_combo_presence(all_games, candidate_tags[:30], combo_size=3)
combo_presence = pd.concat([pair_presence, triple_presence], ignore_index=True)

combo_test = test_binary_presence(combo_presence, "tag_combo", all_games, min_count=50)
combo_test["combo_size"] = combo_test["tag_combo"].str.count(r" \+ ") + 1
combo_test["best_single_response_rate"] = combo_test["tag_combo"].apply(
    lambda combo: max(single_tag_response_rate.get(tag, np.nan) for tag in combo.split(" + "))
)
combo_test["lift_vs_best_single"] = combo_test["response_rate"] / combo_test["best_single_response_rate"]

significant_combo = (
    combo_test
    .query("q_value < 0.05 and diff_pp >= 5 and lift_vs_overall >= 1.1 and response_count >= 30")
    .sort_values(["lift_vs_best_single", "diff_pp", "median_reviews"], ascending=False)
    .reset_index(drop=True)
)

display(
    significant_combo[
        [
            "tag_combo", "combo_size", "game_count", "response_rate", "without_response_rate", "diff_pp",
            "lift_vs_overall", "lift_vs_best_single", "odds_ratio", "median_reviews",
            "median_positive_rate", "q_value", "test",
        ]
    ].head(30).round(3)
)

combo_plot = significant_combo.head(15).sort_values("diff_pp", ascending=True).copy()
combo_plot["label"] = combo_plot.apply(
    lambda row: f"{row['response_rate']:.1f}% / lift {row['lift_vs_best_single']:.2f}x", axis=1
)

fig = px.bar(
    combo_plot,
    x="diff_pp",
    y="tag_combo",
    orientation="h",
    color="combo_size",
    text="label",
    color_continuous_scale="Tealgrn",
    title="리뷰 10개 이상 도달률이 높은 태그 조합",
)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="조합 보유 게임 반응률 - 미보유 게임 반응률 (%p)",
    yaxis_title="태그 조합",
    yaxis=dict(
        tickmode="array",
        tickvals=combo_plot["tag_combo"].tolist(),
        ticktext=combo_plot["tag_combo"].tolist(),
        automargin=True,
    ),
    height=760,
    margin=dict(l=300, t=90, r=190, b=80),
    coloraxis_colorbar_title="조합 크기",
)
fig.show()

pair_combo_rank = significant_combo.query("combo_size == 2").head(15)
triple_combo_rank = significant_combo.query("combo_size == 3").head(15)

print("2개 태그 조합 후보")
display(pair_combo_rank[["tag_combo", "game_count", "response_rate", "diff_pp", "lift_vs_best_single", "median_reviews", "q_value"]].round(3))

print("3개 태그 조합 후보")
display(triple_combo_rank[["tag_combo", "game_count", "response_rate", "diff_pp", "lift_vs_best_single", "median_reviews", "q_value"]].round(3))

,tag_combo,combo_size,game_count,response_rate,without_response_rate,diff_pp,lift_vs_overall,lift_vs_best_single,odds_ratio,median_reviews,median_positive_rate,q_value,test
0,Cozy + Story Rich,2,63,93.651,56.514,37.136,1.653,1.384,10.174,125.0,92.000,0.0,Chi-square
1,Co-op + Rogue-lite,2,97,88.660,56.464,32.196,1.565,1.344,5.800,110.0,84.615,0.0,Chi-square
2,Co-op + Online Co-Op + Rogue-lite,3,50,92.000,56.551,35.449,1.624,1.337,7.939,180.5,80.087,0.0,Chi-square
3,Building + Cozy,2,52,90.385,56.552,33.833,1.595,1.336,6.635,181.5,91.719,0.0,Chi-square
4,Cozy + Sandbox,2,50,90.000,56.558,33.442,1.588,1.330,6.354,195.0,90.934,0.0,Chi-square
5,Building + Early Access + Sandbox,3,51,88.235,56.561,31.674,1.557,1.308,5.376,453.0,81.407,0.0,Chi-square
6,Management + Story Rich,2,99,82.828,56.497,26.331,1.462,1.301,3.630,89.0,85.042,0.0,Chi-square
7,Co-op + Sandbox,2,60,85.000,56.555,28.445,1.500,1.289,4.164,452.0,81.633,0.0,Chi-square
8,Online Co-Op + Rogue-lite,2,61,88.525,56.540,31.985,1.562,1.286,5.586,198.0,80.000,0.0,Chi-square
9,Building + Life Sim + Sandbox,3,59,84.746,56.558,28.187,1.496,1.286,4.083,201.0,84.210,0.0,Chi-square


2개 태그 조합 후보


,tag_combo,game_count,response_rate,diff_pp,lift_vs_best_single,median_reviews,q_value
0,Cozy + Story Rich,63,93.651,37.136,1.384,125.0,0.0
1,Co-op + Rogue-lite,97,88.660,32.196,1.344,110.0,0.0
3,Building + Cozy,52,90.385,33.833,1.336,181.5,0.0
4,Cozy + Sandbox,50,90.000,33.442,1.330,195.0,0.0
6,Management + Story Rich,99,82.828,26.331,1.301,89.0,0.0
7,Co-op + Sandbox,60,85.000,28.445,1.289,452.0,0.0
8,Online Co-Op + Rogue-lite,61,88.525,31.985,1.286,198.0,0.0
10,Isometric + Sandbox,59,83.051,26.486,1.285,67.0,0.0
12,Management + Rogue-lite,70,81.429,24.875,1.279,50.0,0.0
13,Difficult + Story Rich,106,78.302,21.786,1.278,54.0,0.0


3개 태그 조합 후보


,tag_combo,game_count,response_rate,diff_pp,lift_vs_best_single,median_reviews,q_value
2,Co-op + Online Co-Op + Rogue-lite,50,92.000,35.449,1.337,180.5,0.000
5,Building + Early Access + Sandbox,51,88.235,31.674,1.308,453.0,0.000
9,Building + Life Sim + Sandbox,59,84.746,28.187,1.286,201.0,0.000
11,City Builder + Management + Sandbox,83,86.747,30.244,1.282,247.0,0.000
15,Life Sim + Management + Sandbox,56,82.143,25.570,1.271,94.0,0.000
16,Base-Building + Management + Sandbox,100,83.000,26.506,1.261,111.5,0.000
17,Building + Management + Sandbox,141,82.979,26.556,1.260,105.0,0.000
19,Co-op + Early Access + Online Co-Op,56,85.714,29.154,1.246,106.5,0.000
23,Base-Building + Building + Sandbox,169,81.065,24.669,1.230,138.0,0.000
24,Building + Colony Sim + Sandbox,57,92.982,36.451,1.220,263.0,0.000


**해석:** 태그 조합은 `q_value < 0.05`, `diff_pp >= 5%p`, `lift_vs_overall >= 1.1`, `response_count >= 30` 기준을 함께 통과한 후보만 해석한다. 특히 `lift_vs_best_single`이 1보다 크면 조합의 반응률이 조합 안의 가장 강한 단일 태그보다도 높다는 뜻이므로, 단순 인기 태그의 반복이 아니라 포지셔닝 조합 후보로 볼 수 있다.

### 6-5. 가격·출시연도·상점 신뢰 신호 통제 후 점검

태그 조합의 차이가 가격대, 출시연도, 플랫폼 지원, 도전과제, 설명 길이 같은 다른 요인에서 비롯됐을 수 있다. 따라서 상위 조합을 이진 변수로 만들고, 기본 출시 조건을 함께 넣은 로지스틱 회귀로 방향성을 점검한다. 이 모델은 인과 증명이 아니라 “다른 조건을 함께 보아도 조합 신호가 남는가”를 확인하는 보조 분석이다.

In [152]:
top_model_combos = significant_combo.head(15)["tag_combo"].tolist()
model_df = all_games.copy()

for combo in top_model_combos:
    combo_tags = set(combo.split(" + "))
    col = "combo__" + combo.replace(" + ", "__").replace(" ", "_").replace("-", "_")
    model_df[col] = model_df["tag_list"].apply(lambda tags: combo_tags.issubset(set(tags))).astype(int)

combo_cols = [col for col in model_df.columns if col.startswith("combo__")]
model_features = [
    "price", "release_year", "description_len", "tag_count", "platform_count",
    "has_achievements", "has_controller", "has_cloud", "is_single_player", "is_self_published",
] + combo_cols

model_ready = model_df[model_features + ["is_response"]].copy()
model_ready["release_year"] = model_ready["release_year"].fillna(model_ready["release_year"].median())
model_ready["price"] = model_ready["price"].fillna(model_ready["price"].median())
for col in ["description_len", "tag_count", "platform_count"]:
    model_ready[col] = model_ready[col].fillna(0)
for col in ["has_achievements", "has_controller", "has_cloud", "is_single_player", "is_self_published"]:
    model_ready[col] = model_ready[col].astype(int)

numeric_features = ["price", "release_year", "description_len", "tag_count", "platform_count"]
binary_features = [col for col in model_features if col not in numeric_features]

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("binary", "passthrough", binary_features),
    ]
)

logit_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced")),
    ]
)

X = model_ready[model_features]
y = model_ready["is_response"].astype(int)
logit_model.fit(X, y)

feature_names = numeric_features + binary_features
coef = logit_model.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({"feature": feature_names, "coef": coef})
coef_df["odds_ratio"] = np.exp(coef_df["coef"])
coef_df["label"] = coef_df["feature"].str.replace("combo__", "", regex=False).str.replace("__", " + ", regex=False).str.replace("_", " ", regex=False)

combo_coef = (
    coef_df[coef_df["feature"].str.startswith("combo__")]
    .sort_values("odds_ratio", ascending=False)
    .reset_index(drop=True)
)
control_coef = coef_df[~coef_df["feature"].str.startswith("combo__")].sort_values("odds_ratio", ascending=False)

print("상위 태그 조합의 통제 후 방향성")
display(combo_coef[["label", "coef", "odds_ratio"]].round(3))

print("통제 변수 방향성")
display(control_coef[["label", "coef", "odds_ratio"]].round(3))

coef_plot = combo_coef.head(12).sort_values("odds_ratio", ascending=True)
fig = px.bar(
    coef_plot,
    x="odds_ratio",
    y="label",
    orientation="h",
    text=coef_plot["odds_ratio"].map(lambda x: f"{x:.2f}x"),
    color_discrete_sequence=[COLOR_GOOD],
    title="통제 변수 반영 후에도 양의 방향을 보이는 태그 조합",
)
fig.add_vline(x=1, line_dash="dash", line_color=COLOR_MUTED)
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    xaxis_title="로지스틱 회귀 오즈비(1보다 크면 반응 확보 방향)",
    yaxis_title="태그 조합",
    yaxis=dict(
        tickmode="array",
        tickvals=coef_plot["label"].tolist(),
        ticktext=coef_plot["label"].tolist(),
        automargin=True,
    ),
    height=680,
    margin=dict(l=300, t=90, r=150, b=80),
    showlegend=False,
)
fig.show()

상위 태그 조합의 통제 후 방향성


,label,coef,odds_ratio
0,Cozy + Story Rich,1.605,4.979
1,Co op + Early Access,1.542,4.674
2,Management + Rogue lite,1.412,4.106
3,Building + Early Access + Sandbox,1.176,3.243
4,Building + Life Sim + Sandbox,1.175,3.237
5,Cozy + Sandbox,1.098,2.999
6,Co op + Rogue lite,1.092,2.979
7,Co op + Sandbox,1.030,2.801
8,Management + Story Rich,1.011,2.750
9,Isometric + Sandbox,0.942,2.564


통제 변수 방향성


,label,coef,odds_ratio
5,has achievements,0.908,2.481
7,has cloud,0.737,2.090
0,price,0.617,1.853
6,has controller,0.472,1.603
2,description len,0.179,1.196
4,platform count,0.078,1.081
8,is single player,-0.310,0.734
1,release year,-0.416,0.660
9,is self published,-0.711,0.491
3,tag count,-1.016,0.362


**해석:** 회귀 결과에서 오즈비가 1보다 큰 태그 조합은 가격, 출시연도, 설명 길이, 태그 수, 플랫폼 수, 도전과제/컨트롤러/Cloud 같은 신뢰 신호를 함께 고려해도 리뷰 10개 이상 확보 방향으로 남아 있는 후보다. 다만 정규화된 수치형 변수와 정규화되지 않은 이진 조합 변수가 함께 들어가므로, 이 결과는 순위 확정보다 방향성 확인용으로 사용한다.

무반응 게임 내부 빈도만 보면 Singleplayer, Casual, 2D, Action, Adventure처럼 Steam에서 매우 흔한 태그 조합에 많이 몰려 있다. 즉, 특정 태그가 실패 원인이라기보다, 차별화가 약한 일반적 태그 조합만으로는 유저에게 선택될 이유를 만들기 어렵다는 신호로 볼 수 있다.

## 8. 상점 페이지 신뢰 신호 분석

출시 전 유저는 게임을 플레이하기 전에 Steam 페이지의 신뢰 신호를 먼저 본다. 도전과제, 컨트롤러 지원, Steam Cloud, 플랫폼 지원, 설명 길이 등은 구매 전 판단에 영향을 주는 최소 정보다.

In [153]:
trust_features = {
    "도전과제 있음": "has_achievements",
    "컨트롤러 지원": "has_controller",
    "Steam Cloud": "has_cloud",
    "Family Sharing": "has_family_sharing",
    "싱글 플레이": "is_single_player",
    "Mac 지원": "mac",
    "Linux 지원": "linux",
    "개발사=퍼블리셔": "is_self_published",
}

trust_summary = []
for label, col in trust_features.items():
    subset_true = df[df[col]]
    subset_false = df[~df[col]]
    trust_summary.append(
        {
            "feature": label,
            "support_rate": df[col].mean() * 100,
            "median_reviews_with": subset_true["total_reviews"].median() if len(subset_true) else np.nan,
            "median_reviews_without": subset_false["total_reviews"].median() if len(subset_false) else np.nan,
            "zero_rate_with": (subset_true["total_reviews"] == 0).mean() * 100 if len(subset_true) else np.nan,
            "zero_rate_without": (subset_false["total_reviews"] == 0).mean() * 100 if len(subset_false) else np.nan,
        }
    )
trust_summary = pd.DataFrame(trust_summary)
trust_summary["zero_rate_gap"] = trust_summary["zero_rate_without"] - trust_summary["zero_rate_with"]
trust_summary = trust_summary.sort_values("zero_rate_gap", ascending=False)

display(trust_summary.round(2))

fig = px.bar(
    trust_summary.sort_values("zero_rate_gap"),
    x="zero_rate_gap",
    y="feature",
    orientation="h",
    color="zero_rate_gap",
    color_continuous_scale="RdYlGn",
    title="신뢰 신호 부재 시 리뷰 0개 비율이 얼마나 높아지는가",
)
fig.update_layout(xaxis_title="리뷰 0개 비율 차이: 미지원 - 지원 (%p)", yaxis_title="신뢰 신호")
fig.show()

fig = px.box(
    df,
    x="review_band",
    y="description_len",
    category_orders={"review_band": BAND_ORDER},
    color="review_band",
    title="침묵 정도별 상점 설명 길이 분포",
)
fig.update_layout(xaxis_title="침묵 정도", yaxis_title="short_description 길이", showlegend=False)
fig.show()

,feature,support_rate,median_reviews_with,median_reviews_without,zero_rate_with,zero_rate_without,zero_rate_gap
1,컨트롤러 지원,34.75,3.0,3.0,1.72,2.25,0.53
7,개발사=퍼블리셔,81.94,3.0,3.0,2.07,2.07,0.01
6,Linux 지원,10.81,4.0,3.0,2.63,2.00,-0.63
5,Mac 지원,11.71,4.0,3.0,2.81,1.97,-0.85
4,싱글 플레이,97.36,3.0,3.0,2.09,1.14,-0.96
0,도전과제 있음,53.67,4.0,3.0,2.71,1.33,-1.38
3,Family Sharing,99.16,3.0,3.0,2.08,0.00,-2.08
2,Steam Cloud,20.01,4.0,3.0,4.94,1.35,-3.59


**해석:** 신뢰 신호 분석은 “게임이 나쁘다”보다 “구매 전 설득 정보가 충분했는가”를 묻는다. 특정 기능이 없는 게임의 리뷰 0개 비율이 더 높다면, 해당 기능 자체가 원인이라기보다 상점 페이지 완성도와 출시 준비 수준을 나타내는 대리 지표로 해석하는 것이 안전하다.

## 9. 실패 유형 분류

마지막으로 PPT에서 사용할 수 있도록 각 게임을 네 가지 실패 후보 유형으로 분류한다. 이 분류는 원인 확정이 아니라, 출시 전 점검을 위한 진단 프레임이다.

In [154]:
# 데이터 내부 기준으로 상대적 임계값 설정
common_tags = set(tag_summary.head(15)["tag"])
high_price_threshold = df["price"].quantile(0.75)
short_desc_threshold = df["description_len"].quantile(0.25)
low_signal_threshold = 2


def classify_failure_types(row):
    types = []

    if row["total_reviews"] == 0:
        types.append("노출 실패형")

    generic_tag_count = len(set(row["tag_list"]) & common_tags)
    distinctive_tag_count = max(len(row["tag_list"]) - generic_tag_count, 0)
    if generic_tag_count >= 4 and distinctive_tag_count <= 8:
        types.append("차별화 부족형")

    if row["price"] >= high_price_threshold and row["total_reviews"] <= 3:
        types.append("가격/기대 불일치형")

    trust_signal_count = int(row["has_achievements"]) + int(row["has_controller"]) + int(row["has_cloud"]) + int(row["mac"]) + int(row["linux"])
    if trust_signal_count <= low_signal_threshold or row["description_len"] <= short_desc_threshold:
        types.append("신뢰 신호 부족형")

    return types if types else ["경계 반응형"]


failure_type_df = df[["appid", "name", "total_reviews", "price", "review_band", "genres", "tag_list", "description_len"]].copy()
failure_type_df["failure_types"] = df.apply(classify_failure_types, axis=1)
exploded_types = failure_type_df.explode("failure_types")

type_summary = (
    exploded_types.groupby("failure_types")
    .agg(
        game_count=("appid", "count"),
        median_reviews=("total_reviews", "median"),
        median_price=("price", "median"),
        median_description_len=("description_len", "median"),
    )
    .sort_values("game_count", ascending=False)
    .reset_index()
)
type_summary["game_ratio"] = type_summary["game_count"] / len(df) * 100

display(type_summary.round(2))

fig = px.bar(
    type_summary.sort_values("game_count", ascending=True),
    x="game_count",
    y="failure_types",
    orientation="h",
    text=type_summary.sort_values("game_count", ascending=True)["game_ratio"].map(lambda x: f"{x:.1f}%"),
    color="failure_types",
    color_discrete_sequence=[COLOR_RISK, COLOR_ACCENT, COLOR_MAIN, COLOR_MUTED, COLOR_GOOD],
    title="무반응 게임의 실패 후보 유형",
)
fig.update_layout(xaxis_title="게임 수", yaxis_title="실패 후보 유형", showlegend=False)
fig.show()

sample_cases = (
    failure_type_df.assign(failure_types_text=failure_type_df["failure_types"].apply(lambda x: ", ".join(x)))
    .sort_values(["total_reviews", "price"], ascending=[True, False])
    [["appid", "name", "total_reviews", "price", "review_band", "failure_types_text", "genres", "description_len"]]
    .head(20)
)
display(sample_cases)

,failure_types,game_count,median_reviews,median_price,median_description_len,game_ratio
0,신뢰 신호 부족형,5827,3.0,3.99,203.0,87.28
1,차별화 부족형,1057,3.0,2.99,196.0,15.83
2,가격/기대 불일치형,1008,2.0,9.99,220.0,15.10
3,경계 반응형,623,5.0,4.99,231.0,9.33
4,노출 실패형,138,0.0,2.00,186.5,2.07


,appid,name,total_reviews,price,review_band,failure_types_text,genres,description_len
5288,3230870,Archaeology - FROZEN WALL,0,119.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Action', 'Adventure', 'Casual', 'Indie', 'Racing', 'RPG', 'Simulation', 'Sports', 'Strategy']",43
5561,3300350,Campgrounds Adventures: The Big Oopsie,0,24.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Action', 'Adventure', 'Casual', 'Indie', 'Strategy']",254
225,1677980,Unmatched: Digital Edition,0,19.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형","['Casual', 'Indie', 'Simulation', 'Strategy']",225
879,2236300,Grand Emprise: Time Travel Survival,0,19.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Action', 'Adventure', 'Casual', 'Indie', 'RPG', 'Simulation']",290
297,1786580,Halloween Store Simulator,0,14.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Casual', 'Indie', 'Simulation', 'Strategy']",278
576,2088400,Fast Royal,0,14.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Action', 'Indie']",296
2274,2549650,Ammo and Oxygen,0,14.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형","['Action', 'Indie']",263
4974,3151670,Trombone Champ: Unflattened,0,14.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Casual', 'Indie']",298
5211,3211280,Electro Bop Boxing League,0,14.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형","['Action', 'Indie', 'Sports']",286
1051,2271940,Hug Survivor,0,12.99,무반응(리뷰 0개),"노출 실패형, 가격/기대 불일치형, 신뢰 신호 부족형","['Action', 'Casual', 'Indie']",183


**해석:** 실패 유형은 상호 배타적이지 않다. 한 게임이 동시에 노출 실패형이면서 신뢰 신호 부족형일 수 있다. 발표에서는 이 점을 활용해 “실패는 하나의 원인이 아니라 출시 전 포지셔닝, 가격 기대치, 상점 페이지 신뢰 신호가 겹친 결과”라는 결론으로 정리한다.

## 10. PPT용 결론 정리

- 리뷰 10개 미만 게임은 “낮은 평가를 받은 게임”이라기보다 “초기 반응을 충분히 확보하지 못한 게임”이다.
- 실패 원인 분석에서는 긍정률보다 리뷰 수, 리뷰 0개 비율, 장르·태그·가격·상점 페이지 신뢰 신호가 더 중요하다.
- 반응 확보 게임에서는 단일 장르명보다 플레이 루프가 구체적으로 상상되는 태그 조합이 더 강한 후보로 나타나는지 검정해야 한다.
- 무반응 게임은 크게 `노출 실패형`, `차별화 부족형`, `가격/기대 불일치형`, `신뢰 신호 부족형`으로 설명할 수 있다.
- 인디 개발사 입장에서는 출시 전 장르/태그 포지셔닝, 가격 근거, Steam 페이지 완성도, 출시 시점 경쟁도를 사전에 점검해야 한다.

**PPT 메시지:** 실패는 게임성 하나로 설명되지 않는다. 많은 무반응 게임은 유저가 싫어하기 전에, 유저에게 발견되고 선택될 충분한 이유를 만들지 못했다.